In [9]:

from __future__ import annotations

import ast
import csv
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

In [10]:
BASE_DIR = Path(r"D:\AOU-Graduation-Project\BioIntellect\AI\ECG\App\classification")
TRAIN_CSV = BASE_DIR / "ptbxl_train.csv"

# Cap for majority classes. Adjust as needed.
# Classes with fewer samples than their cap keep ALL samples.
CLASS_CAPS: dict[str, int] = {
    "SR":   2500,   # was 13,404 — capped to reduce dominance
    "NORM": 2000,   # was 7,596  — capped to reduce dominance
    "ABQRS": 1500,  # was 2,683  — mild cap for the next largest
    "IMI":  1200,   # was 2,143
    "ASMI": 1200,   # was 1,887
    "LVH":  1200,   # was 1,708
}

# Default cap for all other classes (None = keep all)
DEFAULT_CAP: int | None = None

OUTPUT_INDICES = BASE_DIR / "balanced_train_indices.npy"
OUTPUT_WEIGHTS = BASE_DIR / "sample_weights.npy"

In [11]:
def load_records(csv_path: Path) -> tuple[list[dict], list[set[str]]]:
    """Return (rows, label_sets) where label_sets[i] is the set of SCP codes for row i."""
    rows: list[dict] = []
    label_sets: list[set[str]] = []

    with csv_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
            raw = row.get("scp_codes", "")
            try:
                parsed = ast.literal_eval(str(raw))
                codes = set(str(k) for k in parsed.keys()) if isinstance(parsed, dict) else set()
            except Exception:
                codes = set()
            label_sets.append(codes)

    return rows, label_sets


# ──────────────────────────────────────────────────────────────────────────────
# Step 2: Build per-class sample lists
# ──────────────────────────────────────────────────────────────────────────────

def build_class_index(label_sets: list[set[str]]) -> dict[str, list[int]]:
    """Map each class → list of row indices that contain it."""
    index: dict[str, list[int]] = defaultdict(list)
    for i, codes in enumerate(label_sets):
        for code in codes:
            index[code].append(i)
    return dict(index)


# ──────────────────────────────────────────────────────────────────────────────
# Step 3: Undersample majority classes
# ──────────────────────────────────────────────────────────────────────────────

def undersample(
    label_sets: list[set[str]],
    class_index: dict[str, list[int]],
    class_caps: dict[str, int],
    default_cap: int | None,
    rng: np.random.Generator,
) -> np.ndarray:
    """Return a sorted array of row indices after applying undersampling caps.

    For each class:
      - If a cap is configured and the class has more samples than the cap,
        randomly select `cap` samples from that class.
      - Otherwise keep all samples.

    A row is included if it survives the cap for ANY of its classes.
    """
    # For each capped class, determine which indices are EXCLUDED (dropped)
    # A row is excluded if it belongs to a capped class AND was not selected in the cap.
    excluded_rows: set[int] = set()
    for cls, indices in class_index.items():
        cap = class_caps.get(cls, default_cap)
        if cap is not None and len(indices) > cap:
            chosen = set(rng.choice(indices, size=cap, replace=False).tolist())
            # Any index in this class that wasn't chosen is excluded
            for idx in indices:
                if idx not in chosen:
                    excluded_rows.add(idx)

    # A row is kept only if it was NOT excluded by any capped class
    all_indices = np.arange(len(label_sets), dtype=np.int64)
    kept_rows = np.array([i for i in all_indices if i not in excluded_rows], dtype=np.int64)

    return np.sort(kept_rows)


# ──────────────────────────────────────────────────────────────────────────────
# Step 4: Compute inverse-frequency sample weights
# ──────────────────────────────────────────────────────────────────────────────

def compute_sample_weights(label_sets: list[set[str]]) -> np.ndarray:
    """Compute per-sample weight as the max inverse class frequency across its labels.

    This gives rare-class samples a higher weight in the loss function,
    further correcting any residual imbalance after undersampling.
    """
    n = len(label_sets)
    class_counts = Counter(code for codes in label_sets for code in codes)
    total = n

    weights = np.ones(n, dtype=np.float32)
    for i, codes in enumerate(label_sets):
        if not codes:
            continue
        # Use the maximum inverse frequency among all labels for this sample
        max_weight = max(total / class_counts[c] for c in codes)
        weights[i] = float(max_weight)

    # Normalize to mean=1 so the overall learning rate is unchanged
    weights = weights / weights.mean()
    return weights


# ──────────────────────────────────────────────────────────────────────────────
# Step 5: Threshold optimization
# ──────────────────────────────────────────────────────────────────────────────

def optimize_thresholds(
    y_true: np.ndarray,
    y_scores: np.ndarray,
    class_names: list[str],
    search_steps: int = 50,
) -> np.ndarray:
    """Find per-class decision thresholds that maximize per-class F1.

    Call this AFTER training, on your FULL validation set (not the balanced
    training subset), to find thresholds calibrated to the real class frequency.

    Parameters
    ----------
    y_true   : shape (N, C) — binary ground truth labels
    y_scores : shape (N, C) — model sigmoid outputs
    class_names : list of C class names
    search_steps : number of threshold candidates to try per class

    Returns
    -------
    thresholds : shape (C,) — optimal threshold per class
    """
    n_classes = y_true.shape[1]
    thresholds = np.full(n_classes, 0.5, dtype=np.float32)

    for c in range(n_classes):
        best_f1 = -1.0
        best_thresh = 0.5
        candidates = np.linspace(0.05, 0.95, search_steps)
        gt = y_true[:, c]
        scores = y_scores[:, c]

        for thresh in candidates:
            preds = (scores >= thresh).astype(np.float32)
            tp = float((preds * gt).sum())
            fp = float((preds * (1 - gt)).sum())
            fn = float(((1 - preds) * gt).sum())
            precision = tp / (tp + fp + 1e-8)
            recall = tp / (tp + fn + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = float(thresh)

        thresholds[c] = best_thresh
        print(f"  {class_names[c]:10s}: thresh={best_thresh:.3f}  best_f1={best_f1:.4f}")

    return thresholds

In [12]:
def main() -> None:
    print(f"Loading records from: {TRAIN_CSV}")
    rows, label_sets = load_records(TRAIN_CSV)
    print(f"  Total records : {len(rows)}")

    class_index = build_class_index(label_sets)
    print(f"  Unique classes: {len(class_index)}")

    # Print original class distribution
    print("\nOriginal class distribution (top 15):")
    counts = Counter(code for codes in label_sets for code in codes)
    for code, cnt in counts.most_common(15):
        cap = CLASS_CAPS.get(code, DEFAULT_CAP)
        cap_str = f" -> capped at {cap}" if cap is not None else " -> no cap"
        print(f"  {code:10s}: {cnt:6d} ({cnt/len(rows)*100:.1f}%){cap_str}")

    # Undersample
    rng = np.random.default_rng(seed=42)
    balanced_indices = undersample(
        label_sets=label_sets,
        class_index=class_index,
        class_caps=CLASS_CAPS,
        default_cap=DEFAULT_CAP,
        rng=rng,
    )
    print(f"\nBalanced dataset size: {len(balanced_indices)} records "
          f"(reduced from {len(rows)}, {len(balanced_indices)/len(rows)*100:.1f}% retained)")

    # Print balanced class distribution
    balanced_label_sets = [label_sets[i] for i in balanced_indices]
    balanced_counts = Counter(code for codes in balanced_label_sets for code in codes)
    print("\nBalanced class distribution (top 15):")
    for code, cnt in balanced_counts.most_common(15):
        print(f"  {code:10s}: {cnt:6d} ({cnt/len(balanced_indices)*100:.1f}%)")

    # Compute sample weights on the FULL dataset (used to index later)
    weights = compute_sample_weights(label_sets)
    print(f"\nSample weights: min={weights.min():.3f}  max={weights.max():.3f}  "
          f"mean={weights.mean():.3f}")

    # Save outputs
    np.save(OUTPUT_INDICES, balanced_indices)
    np.save(OUTPUT_WEIGHTS, weights)
    print(f"\nSaved balanced indices -> {OUTPUT_INDICES}")
    print(f"Saved sample weights   -> {OUTPUT_WEIGHTS}")
    print("\nDone. Use balanced_train_indices.npy to filter your training DataFrame.")
    print("Use sample_weights[balanced_train_indices] as Keras sample_weight.")
    print("After training, call optimize_thresholds() on the FULL val set to recalibrate thresholds.")


main()


Loading records from: D:\AOU-Graduation-Project\BioIntellect\AI\ECG\App\classification\ptbxl_train.csv
  Total records : 17418
  Unique classes: 71

Original class distribution (top 15):
  SR        :  13404 (77.0%) -> capped at 2500
  NORM      :   7596 (43.6%) -> capped at 2000
  ABQRS     :   2683 (15.4%) -> capped at 1500
  IMI       :   2143 (12.3%) -> capped at 1200
  ASMI      :   1887 (10.8%) -> capped at 1200
  LVH       :   1708 (9.8%) -> capped at 1200
  NDT       :   1461 (8.4%) -> no cap
  LAFB      :   1298 (7.5%) -> no cap
  AFIB      :   1211 (7.0%) -> no cap
  ISC_      :   1019 (5.9%) -> no cap
  PVC       :    915 (5.3%) -> no cap
  IRBBB     :    894 (5.1%) -> no cap
  STD_      :    807 (4.6%) -> no cap
  VCLVH     :    701 (4.0%) -> no cap
  STACH     :    661 (3.8%) -> no cap

Balanced dataset size: 3745 records (reduced from 17418, 21.5% retained)

Balanced class distribution (top 15):
  SR        :   1243 (33.2%)
  AFIB      :    897 (24.0%)
  NORM      :    57